In [ ]:
import pandas as pd
import numpy as np

cohort = pd.read_pickle(
    "../results/cohort_features_checkpoint.pkl"
)

# All categorical predictors from the verified paper map
all_categorical_cols = [
    "RIAGENDR",          # Gender
    "RIDRETH1",          # Race
    "INDHHINC",          # Income
    "CVDFITLV",          # Cardiovascular fitness level
    "PAQ180",            # Physical activity
    "BPQ020",            # Hypertension history
    "XY_Estimation",     # DNAm estimated sex
    "MCQ160B",           # Heart failure history
    "MCQ160C",           # Coronary artery disease history
    "MCQ160F",           # Cerebrovascular/stroke history
    "DERIVED_HL_MED",    # Hyperlipidemia medication
    "DERIVED_HTN_MED",   # Hypertension medication
    "DERIVED_DM_MED",    # Diabetes medication
    "DIQ010",            # Diabetes history
]

# Only use columns that currently exist
categorical_cols = [
    col for col in all_categorical_cols
    if col in cohort.columns
]

missing_categorical = [
    col for col in all_categorical_cols
    if col not in cohort.columns
]

print("Categorical columns available now:")
print(categorical_cols)

print("\nStill need to build later:")
print(missing_categorical)

Categorical columns available now:
['RIAGENDR', 'RIDRETH1', 'INDHHINC', 'CVDFITLV', 'PAQ180', 'BPQ020', 'MCQ160B', 'MCQ160C', 'MCQ160F', 'DIQ010']

Still need to build later:
['XY_Estimation', 'DERIVED_HL_MED', 'DERIVED_HTN_MED', 'DERIVED_DM_MED']


In [ ]:
cohort_encoded = pd.get_dummies(
    cohort,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 303)


In [ ]:
for col in categorical_cols:
    new_cols = [
        c for c in cohort_encoded.columns
        if c.startswith(col + "_")
    ]

    print(col, "->", new_cols)

RIAGENDR -> ['RIAGENDR_1.0', 'RIAGENDR_2.0']
RIDRETH1 -> ['RIDRETH1_1.0', 'RIDRETH1_2.0', 'RIDRETH1_3.0', 'RIDRETH1_4.0', 'RIDRETH1_5.0']
INDHHINC -> ['INDHHINC_1.0', 'INDHHINC_2.0', 'INDHHINC_3.0', 'INDHHINC_4.0', 'INDHHINC_5.0', 'INDHHINC_6.0', 'INDHHINC_7.0', 'INDHHINC_8.0', 'INDHHINC_9.0', 'INDHHINC_10.0', 'INDHHINC_11.0', 'INDHHINC_12.0', 'INDHHINC_13.0', 'INDHHINC_77.0', 'INDHHINC_99.0']
CVDFITLV -> ['CVDFITLV_1.0', 'CVDFITLV_2.0', 'CVDFITLV_3.0']
PAQ180 -> ['PAQ180_1.0', 'PAQ180_2.0', 'PAQ180_3.0', 'PAQ180_4.0']
BPQ020 -> ['BPQ020_1.0', 'BPQ020_2.0', 'BPQ020_9.0']
MCQ160B -> ['MCQ160B_1.0', 'MCQ160B_2.0', 'MCQ160B_7.0', 'MCQ160B_9.0']
MCQ160C -> ['MCQ160C_1.0', 'MCQ160C_2.0', 'MCQ160C_7.0', 'MCQ160C_9.0']
MCQ160F -> ['MCQ160F_1.0', 'MCQ160F_2.0', 'MCQ160F_9.0']
DIQ010 -> ['DIQ010_1.0', 'DIQ010_2.0', 'DIQ010_3.0', 'DIQ010_7.0', 'DIQ010_9.0']


In [ ]:
cohort_ohe = cohort.copy()

special_missing = {
    "INDHHINC": [77, 99],
    "PAQ180": [7, 9],
    "BPQ020": [7, 9],
    "MCQ160B": [7, 9],
    "MCQ160C": [7, 9],
    "MCQ160F": [7, 9],
    "DIQ010": [7, 9],
}

for col, codes in special_missing.items():
    if col in cohort_ohe.columns:
        cohort_ohe[col] = cohort_ohe[col].replace(codes, np.nan)

In [ ]:
cohort_encoded = pd.get_dummies(
    cohort_ohe,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort_ohe.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 293)


In [ ]:
bad_dummy_cols = [
    c for c in cohort_encoded.columns
    if any(
        c.endswith(suffix)
        for suffix in [
            "_7.0",
            "_9.0",
            "_77.0",
            "_99.0"
        ]
    )
]

print("Invalid dummy columns:", bad_dummy_cols)

Invalid dummy columns: ['INDHHINC_7.0', 'INDHHINC_9.0']


In [ ]:
[c for c in cohort.columns if "mort" in c.lower() or "death" in c.lower()]

['MORTSTAT', 'death_5yr']

In [ ]:
cohort_encoded["mortality_5yr"] = (
    (cohort_encoded["MORTSTAT"] == 1) &
    (cohort_encoded["PERMTH_INT"] <= 60)
).astype(int)

print(cohort_encoded["mortality_5yr"].value_counts())

mortality_5yr
0    8664
1    1293
Name: count, dtype: int64


In [ ]:
y = cohort_encoded["mortality_5yr"]

majority_class = y.mode()[0]

y_pred_stupid = np.full(
    len(y),
    majority_class
)

accuracy = (y_pred_stupid == y).mean()

print("Majority class:", majority_class)
print(f"Stupid model accuracy: {accuracy:.2%}")

Majority class: 0
Stupid model accuracy: 87.01%


In [ ]:
print(
    "Targets identical:",
    (cohort_encoded["death_5yr"] ==
     cohort_encoded["mortality_5yr"]).all()
)

Targets identical: True


In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

y = cohort_encoded["death_5yr"].astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(np.zeros(len(y)), y),
    start=1
):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Learn majority class ONLY from this training fold
    majority_class = y_train.mode().iloc[0]

    # Predict same class for everyone in validation fold
    y_pred = np.full(
        len(y_val),
        majority_class
    )

    accuracy = (
        y_pred == y_val.to_numpy()
    ).mean()

    fold_accuracies.append(accuracy)

    print(
        f"Fold {fold}: "
        f"{accuracy:.2%}"
    )

print(
    "\nMean 5-fold accuracy:",
    f"{np.mean(fold_accuracies):.2%}"
)

print(
    "Std:",
    f"{np.std(fold_accuracies):.4f}"
)

Fold 1: 87.00%
Fold 2: 87.00%
Fold 3: 87.04%
Fold 4: 87.04%
Fold 5: 86.99%

Mean 5-fold accuracy: 87.01%
Std: 0.0002


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np

In [ ]:
y = cohort["death_5yr"].astype(int)

# Never use outcome/linkage fields as predictors
drop_cols = [
    "SEQN",
    "death_5yr",
    "MORTSTAT",
    "PERMTH_INT"
]

X = cohort.drop(
    columns=[c for c in drop_cols if c in cohort.columns]
)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Categorical: 10
Numeric: 251


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    # Learn preprocessing ONLY from training fold
    X_train_processed = preprocessor.fit_transform(X_train)

    # Apply same learned preprocessing to validation
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 2: train = (7965, 297) | validation = (1992, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 3: train = (7966, 297) | validation = (1991, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
all_missing_cols = [
    col for col in X.columns
    if X[col].isna().all()
]

print("Completely empty columns:", len(all_missing_cols))
print(all_missing_cols)

Completely empty columns: 2
['DMDEDUC3', 'DMDSCHOL']


In [ ]:
X = X.drop(columns=all_missing_cols)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Predictors remaining:", X.shape[1])
print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Predictors remaining: 259
Categorical: 10
Numeric: 249


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)
Fold 2: train = (7965, 297) | validation = (1992, 297)
Fold 3: train = (7966, 297) | validation = (1991, 297)
Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


In [1]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Load data
cohort = pd.read_pickle("../results/cohort_features_checkpoint.pkl")

# 2. Select three features
glm_features = ["RIDAGEYR", "SSTNT", "SSBNP"]

X_glm = cohort[glm_features].copy()
y_glm = cohort["death_5yr"].astype(int)

# 3. Build Logistic Regression
glm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# 4. Stratified 5-fold cross-validation
glm_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 5. Evaluate
glm_scores = cross_validate(
    glm,
    X_glm,
    y_glm,
    cv=glm_cv,
    scoring={
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC-AUC": "roc_auc"
    },
    error_score="raise"
)

# 6. Display results
glm_results = pd.DataFrame({
    "Precision": glm_scores["test_Precision"],
    "Recall": glm_scores["test_Recall"],
    "F1": glm_scores["test_F1"],
    "ROC-AUC": glm_scores["test_ROC-AUC"]
})

glm_results.index = range(1, 6)

print("Logistic Regression - 5-Fold Results")
display(glm_results.round(3))

print("\nAverage Performance:")
display(glm_results.mean().round(3))

Logistic Regression - 5-Fold Results


,Precision,Recall,F1,ROC-AUC
1,0.282,0.741,0.409,0.792
2,0.292,0.749,0.420,0.802
3,0.291,0.764,0.421,0.821
4,0.325,0.829,0.467,0.852
5,0.288,0.772,0.420,0.814



Average Performance:


Precision    0.296
Recall       0.771
F1           0.427
ROC-AUC      0.816
dtype: float64

## Random Forest Model

Compare Random Forest with Logistic Regression using
the same three features and stratified 5-fold cross-validation.

In [2]:
from sklearn.ensemble import RandomForestClassifier

# 1. Build Random Forest
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Evaluate using the same 5 folds
rf_scores = cross_validate(
    rf,
    X_glm,
    y_glm,
    cv=glm_cv,
    scoring={
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC-AUC": "roc_auc"
    },
    error_score="raise"
)

# 3. Display Random Forest results
rf_results = pd.DataFrame({
    "Precision": rf_scores["test_Precision"],
    "Recall": rf_scores["test_Recall"],
    "F1": rf_scores["test_F1"],
    "ROC-AUC": rf_scores["test_ROC-AUC"]
})

rf_results.index = range(1, 6)

print("Random Forest - 5-Fold Results")
display(rf_results.round(3))

print("\nAverage Performance:")
display(rf_results.mean().round(3))

# 4. Compare both models
comparison = pd.DataFrame({
    "Logistic Regression": glm_results.mean(),
    "Random Forest": rf_results.mean()
}).T

print("\nModel Comparison:")
display(comparison.round(3))

Random Forest - 5-Fold Results


,Precision,Recall,F1,ROC-AUC
1,0.331,0.629,0.434,0.808
2,0.338,0.695,0.455,0.817
3,0.342,0.729,0.465,0.834
4,0.393,0.771,0.521,0.864
5,0.349,0.710,0.468,0.843



Average Performance:


Precision    0.351
Recall       0.707
F1           0.469
ROC-AUC      0.833
dtype: float64


Model Comparison:


,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.296,0.771,0.427,0.816
Random Forest,0.351,0.707,0.469,0.833


## Comparing Predictions for the Same Participants

In [4]:
import numpy as np
from sklearn.model_selection import cross_val_predict

# Predict probabilities using the same 5 folds
glm_prob = cross_val_predict(
    glm,
    X_glm,
    y_glm,
    cv=glm_cv,
    method="predict_proba"
)[:, 1]

rf_prob = cross_val_predict(
    rf,
    X_glm,
    y_glm,
    cv=glm_cv,
    method="predict_proba"
)[:, 1]

# Compare predictions for every participant
prediction_comparison = X_glm.copy()

prediction_comparison["Actual_Death"] = y_glm.to_numpy()
prediction_comparison["GLM_Probability"] = glm_prob
prediction_comparison["RF_Probability"] = rf_prob

# Absolute difference between the two models
prediction_comparison["Difference"] = np.abs(
    glm_prob - rf_prob
)

# Show participants with the largest differences
print("Participants with largest model disagreement:")

display(
    prediction_comparison
    .sort_values("Difference", ascending=False)
    .head(10)
    .round(3)
)

print(
    "Average probability difference:",
    round(prediction_comparison["Difference"].mean(), 3)
)

Participants with largest model disagreement:


,RIDAGEYR,SSTNT,SSBNP,Actual_Death,GLM_Probability,RF_Probability,Difference
5171,84.0,6.13,57.03,0,0.744,0.112,0.631
9276,82.0,8.81,106.30,0,0.731,0.109,0.622
3009,53.0,143.80,12.30,0,0.966,0.367,0.599
6394,82.0,8.80,158.80,0,0.731,0.134,0.597
180,82.0,8.16,145.80,0,0.734,0.137,0.596
6038,80.0,6.86,49.56,0,0.682,0.090,0.591
1215,79.0,6.34,95.55,1,0.670,0.095,0.575
1998,82.0,7.94,169.00,0,0.722,0.147,0.575
1938,81.0,5.94,49.99,0,0.701,0.127,0.574
6372,82.0,7.47,157.70,0,0.731,0.174,0.557


Average probability difference: 0.148


In [5]:
# Compare yes/no predictions using a 0.5 threshold
prediction_comparison["GLM_Predicted"] = (
    prediction_comparison["GLM_Probability"] >= 0.5
).astype(int)

prediction_comparison["RF_Predicted"] = (
    prediction_comparison["RF_Probability"] >= 0.5
).astype(int)

# Check how often the models disagree
disagree = (
    prediction_comparison["GLM_Predicted"]
    != prediction_comparison["RF_Predicted"]
)

print("Total participants:", len(prediction_comparison))
print("Different classifications:", disagree.sum())
print("Disagreement rate:", round(disagree.mean() * 100, 2), "%")

Total participants: 9957
Different classifications: 1633
Disagreement rate: 16.4 %


In [6]:
# Compare the two models' yes/no predictions
agreement_table = pd.crosstab(
    prediction_comparison["GLM_Predicted"],
    prediction_comparison["RF_Predicted"],
    rownames=["Logistic Regression"],
    colnames=["Random Forest"]
)

print("Model Agreement Table:")
print("0 = Predicts no death")
print("1 = Predicts death")

display(agreement_table)

Model Agreement Table:
0 = Predicts no death
1 = Predicts death


Random Forest,0,1
Logistic Regression,,
0,6149,434
1,1199,2175


## Comparing Individual Feature Explanations

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# Same training and test participants for both models
X_train, X_test, y_train, y_test = train_test_split(
    X_glm,
    y_glm,
    test_size=0.20,
    stratify=y_glm,
    random_state=42
)

# Fit separate copies of our existing models
glm_explain = clone(glm)
rf_explain = clone(rf)

glm_explain.fit(X_train, y_train)
rf_explain.fit(X_train, y_train)

print("Training participants:", len(X_train))
print("Test participants:", len(X_test))
print("Features:", X_train.columns.tolist())
print("Both models are ready for explanation.")

Training participants: 7965
Test participants: 1992
Features: ['RIDAGEYR', 'SSTNT', 'SSBNP']
Both models are ready for explanation.


In [9]:
%pip install "shap>=0.54.1"

  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.6/550.6 kB 4.3 MB/s  0:00:00
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [shap]2/3 [shap]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import shap
import numpy as np
import pandas as pd

# Use the same participants for both models
background = X_train.sample(n=40, random_state=42)
people = X_test.sample(n=20, random_state=42)

# Return predicted probability of death
def glm_probability(data):
    data = pd.DataFrame(data, columns=glm_features)
    return glm_explain.predict_proba(data)[:, 1]

def rf_probability(data):
    data = pd.DataFrame(data, columns=glm_features)
    return rf_explain.predict_proba(data)[:, 1]

# Use the same SHAP method and background for both models
glm_explainer = shap.Explainer(
    glm_probability, background,
    algorithm="permutation", seed=42
)

rf_explainer = shap.Explainer(
    rf_probability, background,
    algorithm="permutation", seed=42
)

# Explain the same 20 held-out participants
glm_shap = glm_explainer(people, max_evals=35)
rf_shap = rf_explainer(people, max_evals=35)

# Find the most influential feature for each participant
glm_top = np.abs(glm_shap.values).argmax(axis=1)
rf_top = np.abs(rf_shap.values).argmax(axis=1)

explanation_comparison = pd.DataFrame({
    "GLM_Top_Feature": [glm_features[i] for i in glm_top],
    "RF_Top_Feature": [glm_features[i] for i in rf_top],
    "Agree": glm_top == rf_top
}, index=people.index)

print("Feature importance comparison:")
display(explanation_comparison)

print(
    "\nTop-feature agreement:",
    round(explanation_comparison["Agree"].mean() * 100, 1),
    "%"
)

Feature importance comparison:


,GLM_Top_Feature,RF_Top_Feature,Agree
1721,RIDAGEYR,SSTNT,False
3385,RIDAGEYR,SSBNP,False
2825,SSTNT,SSBNP,False
6341,RIDAGEYR,SSTNT,False
7290,SSTNT,SSTNT,True
598,RIDAGEYR,RIDAGEYR,True
7668,RIDAGEYR,SSBNP,False
3237,RIDAGEYR,RIDAGEYR,True
6252,RIDAGEYR,RIDAGEYR,True
2116,RIDAGEYR,RIDAGEYR,True



Top-feature agreement: 45.0 %


In [ ]:
### Examining One Participant's SHAP Explanations

In [11]:
# Select one participant where the models disagree
person_id = explanation_comparison.index[
    ~explanation_comparison["Agree"]
][0]

position = people.index.get_loc(person_id)

# Compare the SHAP contributions for each feature
person_explanation = pd.DataFrame({
    "Feature": glm_features,
    "Actual_Value": people.loc[person_id, glm_features].values,
    "GLM_SHAP": glm_shap.values[position],
    "RF_SHAP": rf_shap.values[position]
})

print("Participant index:", person_id)

display(person_explanation.round(4))

# Compare model predictions for the same person
person = people.loc[[person_id]]

print(
    "Logistic Regression prediction:",
    round(glm_explain.predict_proba(person)[0, 1], 3)
)

print(
    "Random Forest prediction:",
    round(rf_explain.predict_proba(person)[0, 1], 3)
)


Participant index: 1721


,Feature,Actual_Value,GLM_SHAP,RF_SHAP
0,RIDAGEYR,60.00,-0.0934,-0.0775
1,SSTNT,5.39,-0.0300,-0.1136
2,SSBNP,11.44,-0.0065,-0.0306


Logistic Regression prediction: 0.293
Random Forest prediction: 0.121


In [12]:
# Test SHAP agreement on 100 held-out participants
people_100 = X_test.sample(n=100, random_state=123)

# Explain both models on exactly the same people
glm_shap_100 = glm_explainer(people_100, max_evals=35)
rf_shap_100 = rf_explainer(people_100, max_evals=35)

# Most influential feature for each person
glm_top_100 = np.abs(glm_shap_100.values).argmax(axis=1)
rf_top_100 = np.abs(rf_shap_100.values).argmax(axis=1)

# Measure agreement
agree_100 = glm_top_100 == rf_top_100

print("Participants tested:", len(people_100))
print("Models agree:", agree_100.sum())
print("Models disagree:", (~agree_100).sum())
print("Agreement rate:", round(agree_100.mean() * 100, 1), "%")
print("Disagreement rate:", round((~agree_100).mean() * 100, 1), "%")

PermutationExplainer explainer: 101it [00:26,  2.33it/s]                         

Participants tested: 100
Models agree: 49
Models disagree: 51
Agreement rate: 49.0 %
Disagreement rate: 51.0 %


In [13]:
# Compare which features each model ranks first

glm_top_features = np.array(glm_features)[glm_top_100]
rf_top_features = np.array(glm_features)[rf_top_100]

feature_agreement_table = pd.crosstab(
    pd.Series(glm_top_features, name="Logistic Regression"),
    pd.Series(rf_top_features, name="Random Forest")
)

print("Top Feature Agreement Table:")
display(feature_agreement_table)

print("\nMost common top feature for Logistic Regression:")
print(pd.Series(glm_top_features).value_counts())

print("\nMost common top feature for Random Forest:")
print(pd.Series(rf_top_features).value_counts())

Top Feature Agreement Table:


Random Forest,RIDAGEYR,SSBNP,SSTNT
Logistic Regression,,,
RIDAGEYR,47,16,34
SSTNT,0,1,2



Most common top feature for Logistic Regression:
RIDAGEYR    97
SSTNT        3
Name: count, dtype: int64

Most common top feature for Random Forest:
RIDAGEYR    47
SSTNT       36
SSBNP       17
Name: count, dtype: int64


### Preliminary Findings

We compared SHAP explanations from Logistic Regression and Random Forest
for 100 held-out NHANES participants.

The models agreed on the most important feature for 49% of participants
and disagreed for 51%.

Logistic Regression mostly ranked age first (97/100), while Random
Forest distributed its top features across age, Troponin-T, and NT-proBNP.

These findings suggest that models with similar predictive performance
may provide different explanations for the same person.

This is a preliminary analysis using three features and 5-year
all-cause mortality.